# Kaggle Training + Evaluation (Run ResNet50 First, VGG19 Optional)

This notebook is intentionally split into separate cells per model.

Recommended flow:
1. Run setup + data prep
2. Run ResNet50 training/evaluation/export cells
3. Only run VGG19 cells if ResNet50 is not good enough


In [ ]:
from pathlib import Path
import os
import shutil

KAGGLE_INPUT = Path('/kaggle/input')
PROJECT_ROOT = Path('/kaggle/working/EYE_HEART_CONNECTION')
ARTIFACT_DIR = Path('/kaggle/working/eye_heart_artifacts')

candidates = list(KAGGLE_INPUT.rglob('full_df.csv'))
if not candidates:
    raise FileNotFoundError('Could not find full_df.csv under /kaggle/input. Upload the full project as a Kaggle Dataset.')

source_root = candidates[0].parent
print(f'Source detected: {source_root}')

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

shutil.copytree(source_root, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
(ARTIFACT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(ARTIFACT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(ARTIFACT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

print('Working directory:', Path.cwd())
print('Artifact directory:', ARTIFACT_DIR)


In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -e .[dev]


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!python -m datasets.build_patient_df --config configs/data.yaml
!python -m datasets.data_quality --data-config configs/data.yaml


In [ ]:
from pathlib import Path
import shutil
import yaml

EPOCHS = 15  # change as needed

with Path('configs/train.yaml').open('r', encoding='utf-8') as f:
    BASE_TRAIN_CFG = yaml.safe_load(f)
with Path('configs/eval.yaml').open('r', encoding='utf-8') as f:
    BASE_EVAL_CFG = yaml.safe_load(f)

def write_train_cfg(model_name: str, epochs: int) -> Path:
    cfg = dict(BASE_TRAIN_CFG)
    cfg['epochs'] = epochs
    cfg['experiment'] = dict(BASE_TRAIN_CFG['experiment'])
    cfg['experiment']['name'] = f'kaggle_{model_name}_{epochs}ep'
    out = Path(f'configs/train_{model_name}_kaggle.yaml')
    with out.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out

def write_eval_cfg(model_name: str) -> Path:
    cfg = dict(BASE_EVAL_CFG)
    cfg['predictions_file'] = f'data/processed/eval_predictions_{model_name}.csv'
    cfg['thresholds_file'] = f'data/processed/thresholds_{model_name}.json'
    cfg['report_file'] = f'data/processed/eval_report_{model_name}.json'
    out = Path(f'configs/eval_{model_name}_kaggle.yaml')
    with out.open('w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out

def export_model_artifacts(model_name: str, run_prefix: str) -> None:
    ckpt_src = Path('experiments/latest/best.pt')
    ckpt_dst = ARTIFACT_DIR / 'models' / f'{model_name}_best.pt'
    if ckpt_src.exists():
        shutil.copy2(ckpt_src, ckpt_dst)

    metric_files = [
        Path(f'data/processed/eval_report_{model_name}.json'),
        Path(f'data/processed/eval_predictions_{model_name}.csv'),
        Path(f'data/processed/thresholds_{model_name}.json'),
    ]
    for src in metric_files:
        if src.exists():
            shutil.copy2(src, ARTIFACT_DIR / 'metrics' / f'{model_name}_{src.name}')

    for run_metrics in Path('experiments').glob(f'{run_prefix}*/metrics.csv'):
        dst = ARTIFACT_DIR / 'logs' / model_name / run_metrics.parent.name
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy2(run_metrics, dst / 'metrics.csv')

print('Helper functions loaded.')


## ResNet50 (Run This First)

In [ ]:
RESNET_MODEL_CFG = Path('configs/model_resnet50.yaml')
RESNET_TRAIN_CFG = write_train_cfg('resnet50', EPOCHS)
RESNET_EVAL_CFG = write_eval_cfg('resnet50')

print('ResNet train cfg:', RESNET_TRAIN_CFG)
print('ResNet eval cfg:', RESNET_EVAL_CFG)


In [ ]:
!python -m training.train --config configs/train_resnet50_kaggle.yaml --data-config configs/data.yaml --model-config configs/model_resnet50.yaml


In [ ]:
!python -m evaluation.run --config configs/eval_resnet50_kaggle.yaml --data-config configs/data.yaml --model-config configs/model_resnet50.yaml --ckpt experiments/latest/best.pt


In [ ]:
export_model_artifacts('resnet50', f'kaggle_resnet50_{EPOCHS}ep')
print('ResNet50 artifacts exported.')


## VGG19 (Optional: Run Only If Needed)

In [ ]:
VGG_MODEL_CFG = Path('configs/model_vgg19.yaml')
VGG_TRAIN_CFG = write_train_cfg('vgg19', EPOCHS)
VGG_EVAL_CFG = write_eval_cfg('vgg19')

print('VGG train cfg:', VGG_TRAIN_CFG)
print('VGG eval cfg:', VGG_EVAL_CFG)


In [ ]:
# RUN THIS ONLY IF YOU WANT TO TRAIN VGG19
!python -m training.train --config configs/train_vgg19_kaggle.yaml --data-config configs/data.yaml --model-config configs/model_vgg19.yaml


In [ ]:
# RUN THIS ONLY IF YOU TRAINED VGG19
!python -m evaluation.run --config configs/eval_vgg19_kaggle.yaml --data-config configs/data.yaml --model-config configs/model_vgg19.yaml --ckpt experiments/latest/best.pt


In [ ]:
# RUN THIS ONLY IF YOU TRAINED VGG19
export_model_artifacts('vgg19', f'kaggle_vgg19_{EPOCHS}ep')
print('VGG19 artifacts exported.')


## Package All Artifacts

In [ ]:
for p in sorted(ARTIFACT_DIR.rglob('*')):
    if p.is_file():
        print(p)

!tar -czf /kaggle/working/eye_heart_artifacts.tar.gz -C /kaggle/working eye_heart_artifacts
print('Archive created: /kaggle/working/eye_heart_artifacts.tar.gz')
